# AI Decathlon Predictor: Trénink modelu strojového učení
Ondřej Krejčí, AI/ML Projekt

Tento notebook dokumentuje kompletní proces vzniku regresního modelu pro predikci bodů v desetiboji. Ukazuje cestu od scrapování dat, přes jejich vyčištění, až po samotný trénink algoritmů a jejich vyhodnocení pomocí standardních metrik. Podrobnější informace se nacházejí na https://github.com/ondrejkrejci1/ML_Project/blob/main/Dokumentace%20ML%20Projektu-%20Krejci.pdf


In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

Knihovny úspěšně načteny.


## 1. Načtení a předzpracování dat
Surová data byla získána web scrapingem z portálu decathlon2000.com. V této fázi je načteme a ukážeme si jejich strukturu. Dále je nutné převést textový zápis běhu na 1500 metrů (např. '4:30.50') na absolutní sekundy, aby s nimi mohl matematický model pracovat.

In [4]:

df = pd.read_csv('data_complete.csv')

display(df.head())

,Name,Points,100m,Long_Jump,Shot_Put,High_Jump,400m,110m_Hurdles,Discus,Pole_Vault,Javelin,1500m,Source_URL
0,Antonio Penalver,8306,11.20,7.55,17.32,2.07,50.68,14.79,47.50,4.8,55.98,271.21,https://www.decathlon2000.com/4634/alhama-de-m...
1,Francisco Javier Benet,7574,11.41,7.09,12.97,1.89,49.99,15.09,41.22,4.8,56.10,272.68,https://www.decathlon2000.com/4634/alhama-de-m...
2,Mark Andrew Bishop,7443,11.08,7.25,12.32,1.92,47.99,14.99,34.62,4.4,54.90,283.28,https://www.decathlon2000.com/4634/alhama-de-m...
3,Antonio Penalver,8534,10.76,7.42,16.50,2.12,49.50,14.32,47.38,5.0,59.32,279.94,https://www.decathlon2000.com/4633/alhama-de-m...
4,Andrei Nazarov,8162,10.88,7.19,13.20,2.12,49.03,14.21,43.24,4.7,58.56,269.57,https://www.decathlon2000.com/4633/alhama-de-m...


## 2. Příprava pro trénink (Train/Test Split)
Nyní rozdělíme data na Features a Target. Dataset následně rozřízneme v poměru 80:20. Na 80 % dat se model bude učit a na zbylých 20 % dat ho následně otestujeme.

In [12]:
disciplines = ['100m', 'Long_Jump', 'Shot_Put', '400m', 'High_Jump']
X = pd.DataFrame(df[disciplines])
y = df['Points']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## 3. Trénování algoritmu (Ensemble Learning)
Pro aplikaci jsem zvolil pokročilou metodu **Gradient Boosting Regressor**. Ta nestaví pouze jeden rozhodovací strom, ale postupně vytváří sérii stromů, kde každý další strom opravuje chyby toho předchozího.

In [13]:
model = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)

model.fit(X_train, y_train)

GradientBoostingRegressor(random_state=42)

## 4. Vyhodnocení přesnosti modelu
Abychom zjistili kvalitu umělé inteligence, necháme ji predikovat výsledky na testovací sadě a porovnáme její odhady s reálnými výsledky. Použijeme k tomu metriky MAE a R-squared (schopnost modelu vysvětlit variabilitu dat).

In [14]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Průměrná absolutní chyba (MAE): {mae:.2f} ")
print(f"Skóre spolehlivosti (R^2): {r2:.4f}")

Průměrná absolutní chyba (MAE): 155.44 bodů
Skóre spolehlivosti (R^2): 0.8798 (maximum je 1.0)


## 5. Export modelu pro webovou aplikaci
Jelikož je model úspěšně natrénován a ověřen, uložíme jej do binárního souboru pomocí knihovny `joblib`. Tento soubor si následně načte webová aplikace ve Streamlitu, takže nemusí tréninkový proces opakovat při každém spuštění.

In [ ]:
joblib.dump(model, 'decathlon_model.joblib')